# Mercari 상품 가격 분석 및 예측

## 02. 데이터 이해

### 2.1 데이터 개요

본 프로젝트에서는 Mercari 상품 데이터를 활용하여 상품 특성과 가격 간의 관계를 분석하고 상품 가격을 예측한다.

데이터는 총 1,482,535개의 상품과 8개의 변수로 구성되어 있으며, 상품명, 상품 상태, 카테고리, 브랜드, 가격, 배송 방식 및 상품 설명 등의 정보를 포함한다.

본 단계에서는 데이터의 구조와 변수 특성, 결측치 현황을 확인하여 이후 분석 및 전처리 방향을 결정한다.

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv(
    "../data/raw/mercari_train.tsv",
    sep="\t"
)

train.head()

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


In [2]:
train.shape

(1482535, 8)

In [3]:
train.info()
# brand_name에 결측이 많다. 
# category_name은 결측률이 적어 결측이 없는 수준
# item_condition_id, shipping 순서형 범주
# category_name, brand_name 범주형 변수
# name, item_description 텍스트 변수

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1482535 entries, 0 to 1482534
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1482535 non-null  int64  
 1   name               1482535 non-null  object 
 2   item_condition_id  1482535 non-null  int64  
 3   category_name      1476208 non-null  object 
 4   brand_name         849853 non-null   object 
 5   price              1482535 non-null  float64
 6   shipping           1482535 non-null  int64  
 7   item_description   1482529 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 90.5+ MB


### 2.2 변수 설명

| 변수 | 데이터 타입 | 변수 설명 | 분석 역할 |
|---|---|---|---|
| `train_id` | int64 | 상품별 고유 식별자 | 식별자 |
| `name` | object | 상품명 | 설명변수 / 텍스트 |
| `item_condition_id` | int64 | 상품의 상태를 나타내는 변수 | 설명변수 / 범주형 |
| `category_name` | object | 상품의 카테고리 | 설명변수 / 범주형 |
| `brand_name` | object | 상품의 브랜드 | 설명변수 / 범주형 |
| `price` | float64 | 상품의 판매 가격 | **목표변수(Target)** |
| `shipping` | int64 | 배송비 부담 여부 | 설명변수 / 범주형 |
| `item_description` | object | 상품 상세 설명 | 설명변수 / 텍스트 |

`price`를 본 프로젝트의 목표변수(Target)로 설정하고, 나머지 상품 특성 변수를 설명변수로 활용한다.

`train_id`는 상품을 식별하기 위한 고유 ID이므로 가격 예측에 직접적인 설명력을 갖는 변수로 판단하여 모델링에서는 제외한다.

`name`과 `item_description`은 텍스트 정보를 포함하고 있으므로 기본 회귀모델에서는 별도로 처리하고, 이후 Feature Engineering 단계에서 텍스트 정보를 활용하는 방안을 검토한다.

`item_condition_id`와 `shipping`은 숫자로 저장되어 있지만 각각 상품 상태와 배송비 부담 여부를 나타내므로 일반적인 연속형 변수와 구분하여 분석한다.

In [4]:
train.describe(include="all")

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
count,1.482535e+06,1482535,1.482535e+06,1476208,849853,1.482535e+06,1.482535e+06,1482529
unique,NaN,1225273,NaN,1287,4809,NaN,NaN,1281423
top,NaN,Bundle,NaN,"Women/Athletic Apparel/Pants, Tights, Leggings",PINK,NaN,NaN,No description yet
freq,NaN,2232,NaN,60177,54088,NaN,NaN,82489
mean,7.412670e+05,NaN,1.907380e+00,NaN,NaN,2.673752e+01,4.472744e-01,NaN
std,4.279711e+05,NaN,9.031586e-01,NaN,NaN,3.858607e+01,4.972124e-01,NaN
min,0.000000e+00,NaN,1.000000e+00,NaN,NaN,0.000000e+00,0.000000e+00,NaN
25%,3.706335e+05,NaN,1.000000e+00,NaN,NaN,1.000000e+01,0.000000e+00,NaN
50%,7.412670e+05,NaN,2.000000e+00,NaN,NaN,1.700000e+01,0.000000e+00,NaN
75%,1.111900e+06,NaN,3.000000e+00,NaN,NaN,2.900000e+01,1.000000e+00,NaN


In [5]:
train.columns

Index(['train_id', 'name', 'item_condition_id', 'category_name', 'brand_name',
       'price', 'shipping', 'item_description'],
      dtype='object')

In [6]:
missing = train.isnull().sum().sort_values(ascending=False)

missing

brand_name           632682
category_name          6327
item_description          6
train_id                  0
item_condition_id         0
name                      0
price                     0
shipping                  0
dtype: int64

### 2.3 결측치 현황

전체 1,482,535개의 관측치를 대상으로 결측치를 확인한 결과, `brand_name`에서 632,682건의 결측치가 발생하여 약 42.7%의 높은 결측률을 보였다. 반면 `category_name`은 약 0.4%, `item_description`은 6건으로 결측 수준이 매우 낮았다.

특히 `brand_name`의 경우 결측 행을 단순히 제거할 경우 전체 데이터의 상당 부분을 손실하게 되므로, 행 삭제 방식은 적절하지 않은 것으로 판단하였다.

또한 브랜드 정보의 결측이 무작위로 발생한 것인지, 또는 브랜드가 없는 상품의 특성을 반영하는 것인지 확인할 필요가 있다. 따라서 이후 가격 분포 및 상품 특성과의 관계를 추가적으로 분석한 후 결측치 처리 방법을 결정한다.

`category_name`과 `item_description`은 결측 비율이 매우 낮기 때문에 이후 분석 목적에 따라 대체 또는 제거가 가능할 것으로 판단된다.

In [7]:
missing_df = pd.DataFrame({
    "결측치": train.isnull().sum(),
    "결측률(%)": train.isnull().mean() * 100
}).sort_values("결측률(%)", ascending=False)

missing_df

,결측치,결측률(%)
brand_name,632682,42.675687
category_name,6327,0.426769
item_description,6,0.000405
train_id,0,0.000000
item_condition_id,0,0.000000
name,0,0.000000
price,0,0.000000
shipping,0,0.000000


### 결측치 분석 결과

변수별 결측치와 결측률을 확인한 결과, `brand_name`에서 632,682건의 결측치가 발생하여 전체 데이터의 약 42.7%를 차지하는 것으로 나타났다.

반면 `category_name`의 결측률은 약 0.4%이며, `item_description`은 6건으로 매우 낮은 수준이었다. `price`를 포함한 나머지 변수에서는 결측치가 확인되지 않았다.

`brand_name`은 결측률이 높기 때문에 단순한 행 삭제 방식으로 처리할 경우 전체 데이터의 상당 부분이 손실될 수 있다. 따라서 해당 변수는 결측 자체가 상품의 특성을 나타낼 가능성을 고려하여 별도의 분석을 수행한 후 처리 방법을 결정한다.

특히 이후 EDA 단계에서 브랜드 정보의 존재 여부에 따른 가격 분포 차이를 확인하여 `brand_name`의 결측 여부가 가격 예측에 유의미한 정보를 제공하는지 검토한다.

In [8]:
train.duplicated().sum()

np.int64(0)

In [9]:
train["train_id"].duplicated().sum()

np.int64(0)

### 중복 데이터 검토

전체 관측치를 대상으로 완전히 동일한 행의 중복 여부를 확인한 결과, 중복 관측치는 존재하지 않았다.

또한 상품별 고유 식별자인 `train_id`의 중복 여부를 확인한 결과 역시 중복값이 발견되지 않았다.

따라서 현재 데이터에서는 동일한 관측치가 반복되거나 하나의 상품 식별자가 여러 관측치에 할당되는 문제는 확인되지 않았으며, 중복 데이터로 인한 분석 편향 가능성은 낮은 것으로 판단하였다.

In [10]:
train["price"].describe()

count    1.482535e+06
mean     2.673752e+01
std      3.858607e+01
min      0.000000e+00
25%      1.000000e+01
50%      1.700000e+01
75%      2.900000e+01
max      2.009000e+03
Name: price, dtype: float64

In [11]:
(train["price"] <= 0).sum()

np.int64(874)

In [12]:
train.loc[train["price"] <= 0, "price"].value_counts().sort_index()

price
0.0    874
Name: count, dtype: int64

### 2.3 데이터 품질 검증

데이터 품질을 확인하기 위해 결측치, 중복 데이터 및 가격 변수의 비정상값을 검토하였다.

#### 결측치

`brand_name`에서 632,682건의 결측치가 발생하여 전체 데이터의 약 42.7%로 가장 높은 결측률을 보였다. 반면 `category_name`은 약 0.4%, `item_description`은 6건으로 결측 수준이 매우 낮았다.

특히 `brand_name`은 결측 행을 단순 제거할 경우 상당한 표본 손실이 발생하므로, 이후 EDA를 통해 브랜드 정보의 존재 여부와 가격 간 관계를 확인한 후 처리 방법을 결정한다.

#### 중복 데이터

전체 행의 중복 여부와 상품 식별자인 `train_id`의 중복 여부를 각각 확인한 결과 모두 0건으로 나타났다.

따라서 동일한 관측치가 반복되거나 상품 식별자가 중복되는 데이터 품질 문제는 확인되지 않았다.

#### 가격 변수의 비정상값

목표변수인 `price`를 확인한 결과 최소값이 0으로 나타났으며, `price <= 0`인 관측치는 총 874건이었다. 해당 관측치를 확인한 결과 모두 `price = 0`이었으며 음수 가격은 존재하지 않았다.

상품의 판매가격이라는 변수 특성을 고려할 때 0원은 일반적인 유효 판매가격으로 보기 어려우므로 비정상값으로 판단하였다.

전체 데이터의 약 0.06%에 해당하는 소수의 관측치이므로, 원본 데이터는 유지하되 이후 전처리 단계에서 해당 관측치를 제외하고 모델링을 진행한다.

#### 가격 분포에 대한 사전 확인

`price`의 평균은 26.74, 중앙값은 17.00으로 평균이 중앙값보다 높게 나타났다. 또한 최대값이 2,009로 확인되어 일부 고가 상품이 존재하는 것으로 나타났다.

이는 가격 분포가 오른쪽으로 치우쳐 있을 가능성을 보여주므로, 이후 EDA 단계에서 가격 분포와 이상치의 형태를 시각적으로 확인하고 필요할 경우 로그 변환 등의 방법을 검토한다.